In [1]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [2]:
# Extract canonical facts from FCA HTML files
from src.parsing.arelle_parser import process_html_files

html_folder = projectRoot / "data" / "raw" / "fca"
output_path = (projectRoot / "data" / "processed" / 
               "canonical_facts" / "bronze.csv")

if output_path.exists():
    print(f"Bronze data already exists at: {output_path}")
    bronze_df = pd.read_csv(output_path)
else:
    bronze_df = process_html_files(html_folder, output_path, debug=False)

sample_bronze = bronze_df.sample(n=250, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_bronze.csv")
sample_bronze.to_csv(csv_path, index=False)
print(f"Bronze sample extract saved to: {csv_path}")

Bronze data already exists at: /workspace/data/processed/canonical_facts/bronze.csv
Bronze sample extract saved to: /workspace/data/processed/debug/sample_bronze.csv


In [3]:
# Create silver ground truth from bronze data
from src.parsing.canonical_facts import create_silver_ground_truth

silver_df = create_silver_ground_truth(bronze_df, 
                        str(projectRoot / "data" / "processed" /
                            "canonical_facts" / "silver.csv"))

sample_silver = silver_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_silver.csv")
sample_silver.to_csv(csv_path, index=False)
print(f"Silver sample extract saved to: {csv_path}")

Created cleaned ground truth with 5174 rows.
Silver data saved to: /workspace/data/processed/canonical_facts/silver.csv
Silver sample extract saved to: /workspace/data/processed/debug/sample_silver.csv


In [4]:
# Create gold ground truth from silver data
from src.parsing.canonical_facts import create_gold_ground_truth

gold_df = create_gold_ground_truth(silver_df, 
                         str(projectRoot / "data" / "processed" /
                              "canonical_facts" / "gold.csv"))

sample_gold = gold_df.sample(n=300, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "sample_gold.csv")
sample_gold.to_csv(csv_path, index=False)
print(f"Gold sample extract saved to: {csv_path}")

Gold data saved to: /workspace/data/processed/canonical_facts/gold.csv
Gold sample extract saved to: /workspace/data/processed/debug/sample_gold.csv


In [5]:
# Generate TINY gold sample
sample_gold_tiny = gold_df[~gold_df['segment'].isin(['Narrative_Disclosure', 
                        'Other_Financial_Metric'])].sample(n=10, random_state=42)
csv_path = (projectRoot / "data" / "processed" /
            "debug" / "tiny_sample_gold.csv")
sample_gold_tiny.to_csv(csv_path, index=False)
print(f"Tiny Gold sample extract saved to: {csv_path}")

Tiny Gold sample extract saved to: /workspace/data/processed/debug/tiny_sample_gold.csv


In [8]:
from src.qa_generation.llm_qa_generator import generate_qa_openai
input_csv_path = (projectRoot / "data" / "processed" /
                    "debug" / "tiny_sample_gold.csv")
output_csv_path = (projectRoot / "data" / "qa" / "debug" /
                   "tiny_sample_qa_pairs.csv")

if output_csv_path.exists():
    print(f"QA pairs already exist at: {output_csv_path}")
    qa_pairs_df = pd.read_csv(output_csv_path)
else:
    qa_pairs_df = generate_qa_openai(input_csv_path, output_csv_path)

QA pairs already exist at: /workspace/data/qa/debug/tiny_sample_qa_pairs.csv


In [10]:
# Evaluate models on QA pairs
base_path = projectRoot / "data" 
input_qa_pairs_path = (base_path / "qa" / "debug" /
                        "tiny_sample_qa_pairs.csv")
evaluation_output_path = (base_path / "results" / "debug" / 
                          "single" / "tiny_sample_llm_evaluation.csv")
raw_output_path = (base_path / "results" / "debug" /
                   "single" / "tiny_sample_llm_raw_outputs.csv")
from src.evaluation.llm_interface import evaluate_with_xbrl_context

evaluation_df, raw_output_df = evaluate_with_xbrl_context(input_qa_pairs_path, 
                                      evaluation_output_path, raw_output_path)

Loaded 10 QA pairs
✓ deepseek-ai/DeepSeek-R1-0528 - Row 0
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 0
✓ deepseek-ai/DeepSeek-R1-0528 - Row 1
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 1
✓ deepseek-ai/DeepSeek-R1-0528 - Row 2
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 2
✓ deepseek-ai/DeepSeek-R1-0528 - Row 3
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 3
✓ deepseek-ai/DeepSeek-R1-0528 - Row 4
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 4

--- Evaluated 5/10 QA pairs ---

✓ deepseek-ai/DeepSeek-R1-0528 - Row 5
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 5
✓ deepseek-ai/DeepSeek-R1-0528 - Row 6
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 6
✓ deepseek-ai/DeepSeek-R1-0528 - Row 7
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 7
✓ deepseek-ai/DeepSeek-R1-0528 - Row 8
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 8
✓ deepseek-ai/DeepSeek-R1-0528 - Row 9
✓ Qwen/Qwen3-235B-A22B-Instruct-2507 - Row 9

--- Evaluated 10/10 QA pairs ---



In [11]:
from src.evaluation.analysis import analyze_results

input_path = projectRoot / "data" / "results" / "debug" / "single"
analyze_results(input_path)


HALLUCINATION BENCHMARK RESULTS

deepseek-ai/DeepSeek-R1-0528:
  RAG Accuracy:             0.0% (with context)
  Knowledge Accuracy:       0.0% (no context)
  Hallucination Rate:     100.0% (confident wrong answers)
  Adversarial Robustness:    0.0% (trusts correct source)
  Hallucination Index:   +0.0%

Qwen/Qwen3-235B-A22B-Instruct-2507:
  RAG Accuracy:            50.0% (with context)
  Knowledge Accuracy:       0.0% (no context)
  Hallucination Rate:      50.0% (confident wrong answers)
  Adversarial Robustness:    0.0% (trusts correct source)
  Hallucination Index:   -50.0%

